# Session 2: Understanding Tool Use — The Agentic Pattern

## Recap
In Session 1, **YOU** decided the query sequence:
1. Search Conditions (SNOMED CT: 44054006) → get patient references
2. Follow references → get Patient demographics
3. Search Observations (HbA1c) → get lab values

Today, the **LLM will make those same decisions**. You'll define the tools
(the same FHIR functions you used before) and ask a clinical question.
The LLM figures out the steps on its own.

## How Tool Use Works
```
You ask a question
       ↓
LLM examines available tools
       ↓
LLM decides: "I should call search_conditions(code='44054006')"
       ↓
YOUR CODE executes the function → returns results
       ↓
LLM reads results → decides next step
       ↓
... repeats until LLM has enough data ...
       ↓
LLM produces a final text answer
```

**Important:** The LLM never executes code directly. It *requests* function
calls, and your code runs them. The LLM plans; the software executes.

### 📋 Clinical Code Reference

| Code | System | Meaning | Used In | ICD-10 Reference |
|------|--------|---------|---------|------------------|
| 44054006 | SNOMED CT | Type 2 Diabetes Mellitus | Condition search | E11 |
| 59621000 | SNOMED CT | Essential Hypertension | (Session 3) | I10 |
| 4548-4 | LOINC | Hemoglobin A1c (HbA1c) | Observation search | - |
| 85354-9 | LOINC | Blood Pressure panel | (Session 3) | - |
| 2160-0 | LOINC | Creatinine [Mass/volume] in Serum or Plasma | (Session 3) | - |

**HbA1c Interpretation:**
- < 5.7%: Normal
- 5.7% – 6.4%: Prediabetes
- ≥ 6.5%: Diabetes
- \> 7.5%: Poor glycemic control — needs intervention

In [ ]:
# Install required packages (only needed once per Colab session)
!pip install -q anthropic requests pandas

In [ ]:
# ============================================================
# SETUP — Anthropic API and FHIR Server
# ============================================================
import os, json, requests
from anthropic import Anthropic

# ---- API Key Setup ----
# Try to get API key from Colab Secrets first, then environment variable
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("ANTHROPIC_API_KEY")

if not api_key:
    raise ValueError("Set ANTHROPIC_API_KEY in Colab Secrets or environment")

# ---- Initialize Anthropic Client ----
client = Anthropic(api_key=api_key)
MODEL = "claude-sonnet-4-20250514"

# ---- FHIR Server ----
FHIR_BASE = "https://launch.smarthealthit.org/v/r4/fhir"

print(f"✅ LLM: Anthropic Claude ({MODEL})")
print(f"✅ FHIR server: {FHIR_BASE}")

In [ ]:

# ============================================================
# FHIR TOOL FUNCTIONS
# ============================================================
# These are the same queries you ran manually in Session 1,
# now packaged as reusable functions.

def search_conditions(code: str, max_results: int = 20) -> dict:
    """Search for Condition resources by diagnosis code (SNOMED CT or ICD-10)."""
    resp = requests.get(f"{FHIR_BASE}/Condition",
        params={"code": code, "_count": max_results, "_format": "json"},
        timeout=15)
    resp.raise_for_status()
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        coding = r.get("code", {}).get("coding", [{}])[0]
        results.append({
            "condition_id": r.get("id", ""),
            "patient_reference": r.get("subject", {}).get("reference", ""),
            "code": coding.get("code", ""),
            "code_display": coding.get("display", ""),
            "onset": r.get("onsetDateTime", "unknown")
        })
    return {"total": bundle.get("total", len(results)), "results": results}


def get_patient(patient_id: str) -> dict:
    """Retrieve a single Patient resource by FHIR ID. Returns demographics."""
    resp = requests.get(f"{FHIR_BASE}/Patient/{patient_id}",
        params={"_format": "json"}, timeout=15)
    resp.raise_for_status()
    p = resp.json()
    name = p.get("name", [{}])[0]
    return {
        "id": p.get("id", patient_id),
        "name": f"{' '.join(name.get('given', []))} {name.get('family', '')}".strip(),
        "birthDate": p.get("birthDate", "unknown"),
        "gender": p.get("gender", "unknown")
    }


def search_observations(patient_id: str, loinc_code: str, max_results: int = 5) -> dict:
    """Search Observations for a patient by LOINC code. Returns values sorted most recent first."""
    resp = requests.get(f"{FHIR_BASE}/Observation",
        params={
            "subject": f"Patient/{patient_id}",
            "code": loinc_code,
            "_sort": "-date",
            "_count": max_results,
            "_format": "json"
        }, timeout=15)
    resp.raise_for_status()
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        value_qty = r.get("valueQuantity", {})
        results.append({
            "date": r.get("effectiveDateTime", "unknown"),
            "value": value_qty.get("value", "N/A"),
            "unit": value_qty.get("unit", "")
        })
    return {"patient_id": patient_id, "loinc_code": loinc_code, "results": results}


# Quick smoke test
print("🔍 Testing FHIR tools...")
test = search_conditions("44054006", max_results=3)
print(f"   search_conditions('44054006'): {test['total']} total, {len(test['results'])} returned")
if test["results"]:
    test_pid = test["results"][0]["patient_reference"].split("/")[-1]
    test_pt = get_patient(test_pid)
    print(f"   get_patient('{test_pid}'): {test_pt['name']}")
    test_obs = search_observations(test_pid, "4548-4", max_results=1)
    print(f"   search_observations('{test_pid}', '4548-4'): {len(test_obs['results'])} results")
print("✅ All FHIR tools working")


In [ ]:
# ============================================================
# TOOL SCHEMAS — Define available tools for the agent
# ============================================================
# Tool schemas tell Claude what functions are available and how to use them.
# Claude uses Anthropic's native tool format.

tools = [
    {
        "name": "search_conditions",
        "description": "Search for patient Condition resources on the FHIR server by diagnosis code (SNOMED CT or ICD-10). Returns a list of conditions with patient references, codes, and onset dates. Use this to find patients with a specific diagnosis.",
        "input_schema": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "Diagnosis code to search for. Examples: '44054006' for Type 2 diabetes (SNOMED CT), '59621000' for hypertension (SNOMED CT). Also accepts ICD-10 codes like 'E11' or 'I10'."
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of condition entries to return. Default 20.",
                    "default": 20
                }
            },
            "required": ["code"]
        }
    },
    {
        "name": "get_patient",
        "description": "Retrieve a single Patient resource by their FHIR patient ID. Returns demographics including full name, birth date, and gender. Use this after getting a patient reference from another resource.",
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The FHIR Patient resource ID (the part after 'Patient/' in a reference). Example: 'abc123'"
                }
            },
            "required": ["patient_id"]
        }
    },
    {
        "name": "search_observations",
        "description": "Search for Observation resources (lab results, vital signs) for a specific patient by LOINC code. Returns values sorted by date with most recent first. Common LOINC codes: '4548-4' for HbA1c, '85354-9' for blood pressure, '2160-0' for creatinine.",
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The FHIR Patient ID to search observations for"
                },
                "loinc_code": {
                    "type": "string",
                    "description": "LOINC code for the observation type. Examples: '4548-4' (HbA1c), '85354-9' (blood pressure), '2160-0' (creatinine)"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return. Default 5.",
                    "default": 5
                }
            },
            "required": ["patient_id", "loinc_code"]
        }
    }
]

available_functions = {
    "search_conditions": search_conditions,
    "get_patient": get_patient,
    "search_observations": search_observations,
}

print("✅ Tool schemas defined (3 tools)")
print()
for t in tools:
    params = ", ".join(t["input_schema"].get("required", []))
    print(f"   • {t['name']}({params})")
    print(f"     {t['description'][:80]}...")
    print()

### 🔍 Understanding Tool Schemas

We just defined **tool schemas** — JSON descriptions that tell the LLM:
- **What** functions are available (the `name`)
- **What** each function does (the `description`)
- **What** arguments each takes (the `parameters`)

The LLM reads these descriptions and decides:
1. Which function to call
2. What arguments to pass
3. When to stop and give a final answer

**The LLM never sees the Python code** — only the descriptions.
This is why good descriptions are critical.

## ✏️ Predict the Agent's Behavior

Before we run the agent, predict what it will do.

**Question the agent will receive:**
*"Find patients with Type 2 diabetes and their most recent HbA1c values.
Which patients have poor glycemic control (HbA1c > 7.5%)?"*

**Given the 3 available tools, predict the sequence of calls:**

1. First call — Tool name? Arguments?

   YOUR PREDICTION:

2. What information from call #1 does the agent need for call #2?

   YOUR PREDICTION:

3. Second set of calls — Tool name? How many times?

   YOUR PREDICTION:

4. Third set of calls — Tool name? Arguments?

   YOUR PREDICTION:

5. After all calls, what should the agent do?

   YOUR PREDICTION:

In [ ]:
# ============================================================
# SYSTEM PROMPT — Tells the agent HOW to think
# ============================================================

SYSTEM_PROMPT = """You are a clinical data assistant with access to a FHIR server
containing synthetic patient data.

When asked a clinical question, use the available tools to query the FHIR server
and build up the data needed to answer. Think step by step:

1. First, identify what diagnosis or condition is relevant and search for it
2. Extract patient references from the conditions found
3. Retrieve patient demographics (name, birthdate, gender) for each patient
4. Look up relevant observations (lab values, vitals) for each patient
5. Synthesize a clear, accurate summary based ONLY on the data you retrieved

Rules:
- NEVER invent or assume data that was not returned by a tool call
- If a query returns no results, state that explicitly
- Always identify patients by name when demographics are available
- When comparing values to clinical thresholds, show the actual values
- Be concise but thorough — include all relevant findings"""

print("📝 System prompt defined")
print()
print("--- SYSTEM PROMPT ---")
print(SYSTEM_PROMPT)


In [ ]:
# ============================================================
# AGENT LOOP — Run tool-use conversation with Claude
# ============================================================

def run_agent(question, system_prompt, tools, available_functions,
              max_steps=15, verbose=True):
    """
    Run the tool-use agent loop with Claude.

    Args:
        question: The clinical question to answer
        system_prompt: Instructions for the agent
        tools: Tool schemas in Anthropic format
        available_functions: Dict mapping function names to callables
        max_steps: Safety limit on LLM round-trips
        verbose: Print trace output

    Returns:
        (final_answer, tool_calls_log, messages)
    """
    tool_calls_log = []
    step = 0
    messages = [{"role": "user", "content": question}]

    if verbose:
        print(f"🧑‍⚕️ QUESTION: {question}\n")
        print("=" * 70)

    while step < max_steps:
        step += 1
        
        # Call Claude with tools
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        # Check if Claude wants to use tools or provide final answer
        tool_use_blocks = [b for b in response.content if b.type == "tool_use"]
        text_blocks = [b for b in response.content if b.type == "text"]

        if not tool_use_blocks:
            # No more tool calls — Claude has the final answer
            final = "\n".join(b.text for b in text_blocks)
            if verbose:
                print(f"\n{'=' * 70}")
                print(f"✅ FINAL ANSWER ({len(tool_calls_log)} tool calls):\n")
                print(final)
            return final, tool_calls_log, messages

        # Claude wants to use tools — serialize content blocks to avoid SDK issues
        assistant_content = []
        for block in response.content:
            if block.type == "text":
                assistant_content.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                assistant_content.append({
                    "type": "tool_use",
                    "id": block.id,
                    "name": block.name,
                    "input": block.input
                })
        
        messages.append({"role": "assistant", "content": assistant_content})
        tool_results = []

        for block in tool_use_blocks:
            fn_name = block.name
            fn_args = block.input
            tool_calls_log.append({
                "step": step,
                "function": fn_name,
                "arguments": fn_args
            })

            if verbose:
                print(f"\n🔧 Step {step} | {fn_name}({json.dumps(fn_args)})")

            try:
                # Execute the function
                result = available_functions[fn_name](**fn_args)
                result_str = json.dumps(result, default=str)
                
                if verbose:
                    n_items = len(result.get("results", [])) if isinstance(result, dict) and "results" in result else None
                    if n_items is not None:
                        print(f"   → {n_items} items returned")
                    else:
                        preview = result_str[:100]
                        print(f"   → {preview}...")
            except Exception as e:
                result_str = json.dumps({"error": str(e)})
                if verbose:
                    print(f"   → ❌ Error: {e}")

            # Add tool result for Claude
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result_str
            })

        # Send tool results back to Claude
        messages.append({"role": "user", "content": tool_results})

    # Safety: max steps reached without final answer
    if verbose:
        print(f"\n⚠️ Reached maximum steps ({max_steps}) without final answer")
    return "Max steps reached without final answer", tool_calls_log, messages


print("✅ Agent loop ready")

## 🚀 Run the Agent

The cell below runs the full agent loop on the **same clinical question** from Session 1.
Watch the trace — you should recognize every step.

In [ ]:
# ============================================================
# RUN THE AGENT — Same question as Session 1
# ============================================================
user_question = (
    "Find patients with Type 2 diabetes and their most recent HbA1c values. "
    "Which patients have poor glycemic control (HbA1c > 7.5%)?")

final_answer, tool_calls_log, messages = run_agent(
    question=user_question,
    system_prompt=SYSTEM_PROMPT,
    tools=tools,
    available_functions=available_functions
)

In [ ]:
# ============================================================
# TOOL CALL TRACE ANALYSIS
# ============================================================
print("📋 TOOL CALL SEQUENCE\n")
print(f"{'Step':<6} {'Function':<25} {'Key Arguments'}")
print("-" * 70)

for tc in tool_calls_log:
    args_summary = ", ".join(f"{k}={v}" for k, v in tc["arguments"].items())
    print(f"{tc['step']:<6} {tc['function']:<25} {args_summary}")

print(f"\n{'=' * 70}")
call_types = [tc["function"] for tc in tool_calls_log]
unique_tools = set(call_types)
print(f"Total tool calls: {len(tool_calls_log)}")
print(f"Unique tools used: {len(unique_tools)} — {', '.join(sorted(unique_tools))}")
for fn in sorted(unique_tools):
    print(f"   • {fn}: called {call_types.count(fn)} time(s)")


## ✏️ Compare to Your Predictions

Look at the trace above and answer:

1. Did the agent follow the sequence you predicted? What was different?

   YOUR ANSWER:

2. How did the agent know to use LOINC code `4548-4` for HbA1c?
   (Hint: look at the tool schema descriptions)

   YOUR ANSWER:

3. The agent called `get_patient` multiple times — once per patient.
   Why couldn't it get all patients in a single call?
   (Hint: look at the `get_patient` function signature)

   YOUR ANSWER:

4. What would happen if we REMOVED the system prompt? Would the agent still work?

   YOUR ANSWER:

In [ ]:
# ============================================================
# FULL CONVERSATION ANATOMY
# ============================================================
# This shows every message exchanged between Claude and the tools.
# In a real app, the user only sees the final answer.

print("🔬 FULL CONVERSATION\n")
print("=" * 70)

for i, m in enumerate(messages):
    role = m.get('role', '?')
    content = m.get('content', '')

    if role == 'user':
        if isinstance(content, str):
            print(f"\n[{i}] 🧑‍⚕️ USER: {str(content)[:120]}...")
        elif isinstance(content, list):
            # Tool results (list of tool_result blocks)
            print(f"\n[{i}] 🔧 TOOL RESULTS ({len(content)} result(s))")
        else:
            print(f"\n[{i}] 🧑‍⚕️ USER: {str(content)[:120]}")
    elif role == 'assistant':
        # Check for Anthropic content blocks
        if isinstance(content, list):
            for block in content:
                if hasattr(block, 'type'):
                    if block.type == 'tool_use':
                        print(f"\n[{i}] 🤖 ASSISTANT → tool call: {block.name}(...)")
                    elif block.type == 'text' and block.text:
                        print(f"\n[{i}] 🤖 ASSISTANT: {block.text[:150]}...")
        elif content:
            c = content if isinstance(content, str) else str(content)
            print(f"\n[{i}] 🤖 ASSISTANT: {c[:150]}...")

print(f"\n{'=' * 70}")
print(f"Total messages: {len(messages)}")

## 🧠 Session 2 Takeaways

**What you saw:**
- The LLM received your question and a menu of available tools (schemas)
- It *planned* a sequence of tool calls — the same sequence you executed manually in Session 1
- After each tool call, it received results and decided what to do next
- When it had enough data, it synthesized a final answer

**The agent loop pattern:**
1. User asks question
2. LLM picks a tool + arguments → sends back a `tool_call` request
3. Your code executes the function → sends results back
4. Repeat until LLM produces a text response

**Critical design decisions:**
- **System prompt** guided the agent's strategy
- **Tool descriptions** told the agent what each function does
- **Your code** maintained control — the LLM never accessed the FHIR server directly

**Next session:** You'll pose your OWN clinical questions with MORE tools
available and watch the agent adapt — or fail.